In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import session_info
import matplotlib.pyplot as plt
import seaborn as sns

/home/Data_Drive_8TB/ahwodn/.local/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/Data_Drive_8TB/ahwodn/.local/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/Data_Drive_8TB/ahwodn/.local/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/Data_Drive_8TB/ahwodn/.local/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/Data_Drive_8TB/ahwodn/.local/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Impor

In [2]:
session_info.show()

In [3]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.settings.set_figure_params(dpi=300, facecolor='white', color_map = 'RdPu',)

In [ ]:
adata=sc.read_h5ad('/path/to/h5ad_annotated/_sc_Epithelial.h5ad')

# 1. ORA

In [4]:
rgg_key = "rank_genes_groups"  
groupby_key = "sub_anno"

# 1) DEG -> gene_name_dict 생성
group_names = adata.uns[rgg_key]["names"].dtype.names
gene_name_dict = {}

for group in group_names:
    df = sc.get.rank_genes_groups_df(adata, group=group, key=rgg_key)
    df_filtered = df[(df["pvals_adj"] < 0.05) & (df["logfoldchanges"] > 1.5)] 
    genes = df_filtered["names"].dropna().astype(str).unique().tolist()
    if genes:
        gene_name_dict[str(group)] = genes

In [5]:
#itaiyanai epithelial state

xlsx_path = "Epithelial_state.xlsx"

df = pd.read_excel(xlsx_path)  # header 지정하지 않음

Epithelial_state = {
    col: (
        df[col].dropna().astype(str).str.strip()
        .loc[lambda s: (s != "") & (s.str.lower() != "nan")]
        .drop_duplicates()
        .tolist()
    )
    for col in df.columns
}

In [6]:
import gseapy as gp
from gseapy import enrichr

In [7]:
# 2) Enrichr 실행
gene_sets_list = [
    "MSigDB_Hallmark_2020",
    Epithelial_state,   # dict
]

# 라이브러리 존재 확인: 문자열 gene set만 체크
available = set(gp.get_library_name(organism="Mouse"))
missing = [gs for gs in gene_sets_list if isinstance(gs, str) and gs not in available]
if missing:
    raise ValueError(f"Mouse Enrichr에 없는 gene set: {missing}")

results = []
for gene_set in gene_sets_list:
    gene_set_name = "Epithelial_state" if isinstance(gene_set, dict) else gene_set

    for cluster, gene_list in gene_name_dict.items():
        enr = gp.enrichr(
            gene_list=gene_list,      
            gene_sets=gene_set, 
            outdir=None,
            no_plot=True,
            verbose=False,
        )

        res_raw = enr.results
        if isinstance(res_raw, pd.DataFrame):
            res = res_raw.copy()
        elif isinstance(res_raw, list):
            if len(res_raw) == 0:
                continue
            res = pd.DataFrame(res_raw)
        else:
            continue

        if res.empty:
            continue

        res["cluster_name"] = cluster
        res["Gene_set"] = gene_set_name
        res["n_query_genes"] = len(gene_list)
        results.append(res)

final_results_df = pd.concat(results, ignore_index=True) if results else pd.DataFrame()
print(final_results_df.shape)
final_results_df.head()


(864, 12)


,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,cluster_name,n_query_genes
0,MSigDB_Hallmark_2020,E2F Targets,110/200,1.511165e-132,6.044661e-131,0.0,0.0,73.933057,22440.778977,DCLRE1B;DSCC1;CSE1L;BUB1B;MKI67;SMC4;CDC20;PTT...,Epi_Cycle,432
1,MSigDB_Hallmark_2020,G2-M Checkpoint,88/200,7.249505e-94,1.449901e-92,0.0,0.0,44.438538,9530.380694,TOP2A;SUV39H1;CCNF;HMGB3;KIF11;MKI67;BRCA2;SMC...,Epi_Cycle,432
2,MSigDB_Hallmark_2020,Mitotic Spindle,37/199,4.945603e-24,6.594137e-23,0.0,0.0,11.220847,602.150410,TOP2A;PIF1;TTK;KIF11;BRCA2;SMC4;KIF15;LMNB1;AU...,Epi_Cycle,432
3,MSigDB_Hallmark_2020,DNA Repair,19/150,6.308342e-10,6.308342e-09,0.0,0.0,6.825925,144.600237,RFC5;FEN1;RFC3;DUT;RFC4;PCNA;LIG1;RFC2;ALYREF;...,Epi_Cycle,432
4,MSigDB_Hallmark_2020,Spermatogenesis,17/135,5.549440e-09,4.439552e-08,0.0,0.0,6.752093,128.354379,RPL39L;RFC4;YBX2;TTK;NCAPH;AURKA;CCNB2;DBF4;DM...,Epi_Cycle,432


In [8]:
final_results_df_dd=final_results_df[final_results_df['Adjusted P-value'] < 0.05]

In [9]:
final_results_df_dd = final_results_df_dd.copy()
final_results_df_dd["-log10(Adjusted P-value)"] = -np.log10(
    pd.to_numeric(final_results_df_dd["Adjusted P-value"], errors="coerce").clip(lower=1e-300)
)

In [10]:
final_results_df_dd

,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,cluster_name,n_query_genes,-log10(Adjusted P-value)
0,MSigDB_Hallmark_2020,E2F Targets,110/200,1.511165e-132,6.044661e-131,0.0,0.0,73.933057,22440.778977,DCLRE1B;DSCC1;CSE1L;BUB1B;MKI67;SMC4;CDC20;PTT...,Epi_Cycle,432,130.218628
1,MSigDB_Hallmark_2020,G2-M Checkpoint,88/200,7.249505e-94,1.449901e-92,0.0,0.0,44.438538,9530.380694,TOP2A;SUV39H1;CCNF;HMGB3;KIF11;MKI67;BRCA2;SMC...,Epi_Cycle,432,91.838662
2,MSigDB_Hallmark_2020,Mitotic Spindle,37/199,4.945603e-24,6.594137e-23,0.0,0.0,11.220847,602.150410,TOP2A;PIF1;TTK;KIF11;BRCA2;SMC4;KIF15;LMNB1;AU...,Epi_Cycle,432,22.180842
3,MSigDB_Hallmark_2020,DNA Repair,19/150,6.308342e-10,6.308342e-09,0.0,0.0,6.825925,144.600237,RFC5;FEN1;RFC3;DUT;RFC4;PCNA;LIG1;RFC2;ALYREF;...,Epi_Cycle,432,8.200085
4,MSigDB_Hallmark_2020,Spermatogenesis,17/135,5.549440e-09,4.439552e-08,0.0,0.0,6.752093,128.354379,RPL39L;RFC4;YBX2;TTK;NCAPH;AURKA;CCNB2;DBF4;DM...,Epi_Cycle,432,7.352661
...,...,...,...,...,...,...,...,...,...,...,...,...,...
822,Epithelial_state,Stress,44/233,1.823468e-04,1.185254e-03,NaN,NaN,2.144254,18.461166,FOS;FOSB;ARID5B;DNAJB4;CA12;EDN1;GADD45B;PDK4;...,Epi_Hypox-Adapt,903,2.926188
827,Epithelial_state,Ciliated,32/39,1.641692e-30,2.134199e-29,NaN,NaN,69.829517,4789.035726,DYNLL1;PSENEN;FAM183A;PIFO;LRRIQ1;MS4A8;C1orf1...,Epi_Ciliated,1418,28.670765
846,Epithelial_state,NPC,6/30,1.578728e-03,2.210219e-02,NaN,NaN,6.181918,39.880393,CCNG2;CELF5;ARL4D;CAMK2N1;RAB3A;CELF4,Epi_Oxphos-Ion,887,1.655565
858,Epithelial_state,Metal,4/9,3.047263e-04,3.656715e-03,NaN,NaN,19.879121,160.943283,MT1G;MT1M;MT1F;MT1H,Epi_Oxphos-Metal,812,2.436909


In [11]:
from plotting import (apply_style, pval_to_stars, row_zscore,
                          plot_enrichment_clustermap_combined_score_dif)

apply_style()

In [12]:
dd_sub=adata.copy()

In [13]:
sns.set_style("white")

plt.rcParams["font.family"] = "Arial"
plt.rcParams["mathtext.fontset"] = "custom"   # 수식($...$)도 Arial로
plt.rcParams["mathtext.rm"] = "Arial"
plt.rcParams["mathtext.it"] = "Arial:italic"
plt.rcParams["mathtext.bf"] = "Arial:bold"
plt.rcParams["pdf.fonttype"] = 42             # PDF에 텍스트를 편집 가능한 형태로 임베드 (저널 제출용)



state_order = list(dd_sub.obs["sub_anno"].cat.categories)

cluster_color_map = dict(
    zip(dd_sub.obs["sub_anno"].cat.categories, dd_sub.uns["sub_anno_colors"])
)

cluster_color_map = {k: cluster_color_map[k] for k in state_order}

final_results_df_dd2 = final_results_df_dd.copy()

g, plot_df, row_meta, raw_score_df = plot_enrichment_clustermap_combined_score_dif(
    final_results_df_dd=final_results_df_dd2,
    gene_sets_to_use=["MSigDB_Hallmark_2020"],
    cluster_order=state_order,
    top_k=3,
    score_col='-log10(Adjusted P-value)',
    cbar_label=r'-log$_{10}$(Adjusted $P$-value)',
    title='MSigDB Hallmark 2020',
    use_row_zscore=False,
    cluster_color_map=cluster_color_map,
    col_colors_gap=0.002,
    col_colors_ratio=0.03,
    figsize=(34, 35),
    vmax=10,
    cmap='RdPu_r',
    dpi=700,
    save_path="./figures/hallmark_ora_specificity_top3.pdf",
)


/tmp/ipykernel_1701117/726880703.py:95: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df.pivot_table(
/tmp/ipykernel_1701117/726880703.py:106: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df.pivot_table(


In [14]:
state_order = list(dd_sub.obs["sub_anno"].cat.categories)

cluster_color_map = dict(
    zip(dd_sub.obs["sub_anno"].cat.categories, dd_sub.uns["sub_anno_colors"])
)

cluster_color_map = {k: cluster_color_map[k] for k in state_order}

final_results_df_dd2 = final_results_df_dd.copy()

g, plot_df, row_meta, raw_score_df = plot_enrichment_clustermap_combined_score_dif(
    final_results_df_dd=final_results_df_dd2,
    gene_sets_to_use=["Epithelial_state"],
    cluster_order=state_order,
    top_k=3,
    score_col='-log10(Adjusted P-value)',
    cbar_label=r'-log$_{10}$(Adjusted $P$-value)',
    title='Barkley, D. et. al.(2022)',
    use_row_zscore=False,
    cluster_color_map=cluster_color_map,
    col_colors_ratio=0.03,
    col_colors_gap=0.002,
    figsize=(24, 25),
    vmax=10,
    cmap='RdPu_r',
    dpi=700,
    save_path="./figures/Barkley_ora_specificity_top3.pdf",
)


/tmp/ipykernel_1701117/726880703.py:95: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df.pivot_table(
/tmp/ipykernel_1701117/726880703.py:106: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df.pivot_table(
